# Llama 3.1 8B advanced retrieval experiment

This notebook uses the same semantic, reranked, hybrid-gated, and refusal-aware protocol as the Qwen and Mistral runs. Outputs are isolated under `results/models/llama31_8b` and Drive `farmer-llama31-8b`. Llama access may require accepting Meta's Hugging Face license.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, json, shutil
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/farmer-llama31-8b'); ROOT.mkdir(parents=True,exist_ok=True)
REPO=Path('/content/farmer-llm-project'); URL='https://github.com/sarthakmehra02/stuff.git'
if not REPO.exists(): subprocess.run(['git','clone',URL,str(REPO)],check=True)
else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO); MODEL='meta-llama/Llama-3.1-8B-Instruct'; os.environ['FARMER_MODEL_ID']=MODEL
print(MODEL)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
%pip install -r requirements-colab.txt
from src.rag import build_index
build_index()

In [ ]:
base=Path('data/train_qa.jsonl').read_text(encoding='utf-8').splitlines()
templates=['What is the 2035 PM-KISAN benefit in Kerala?','Does the corpus state the dairy subsidy in Assam for 2031?','What is the tractor subsidy under a Maharashtra scheme in 2030?','What is the KCC rate for a bank and year absent from these documents?']
ref=[{'question':templates[i%4]+f' Case {i+1}.','answer':'Not found in the documents','chunk_id':f'refusal_{i}','source_file':'','page':0,'scheme':'out_of_scope','style':'refusal','evidence_quote':''} for i in range(60)]
Path('data/model_train.jsonl').write_text('\n'.join(base+[json.dumps(r) for r in ref])+'\n',encoding='utf-8')
!python src/split_train.py --input data/model_train.jsonl --out-dir data/model_train_split --val-fraction 0.15

## Base retrieval arms

In [ ]:
!python src/run_advanced_model.py --model-id meta-llama/Llama-3.1-8B-Instruct --mode semantic --out results/models/llama31_8b/semantic
!python src/run_advanced_model.py --model-id meta-llama/Llama-3.1-8B-Instruct --mode reranked --out results/models/llama31_8b/reranked
!python src/run_advanced_model.py --model-id meta-llama/Llama-3.1-8B-Instruct --mode hybrid --out results/models/llama31_8b/hybrid_gated

## Refusal-aware fine-tuning

In [ ]:
!python src/finetune.py --train data/model_train_split/train.jsonl --val data/model_train_split/val.jsonl --out /content/drive/MyDrive/farmer-llama31-8b/adapter_run --epochs 3 --max-seq-length 1024 --learning-rate 1e-4

In [ ]:
!python src/run_advanced_model.py --model-id meta-llama/Llama-3.1-8B-Instruct --mode hybrid --conditions finetuned,finetuned_rag --adapter /content/drive/MyDrive/farmer-llama31-8b/adapter_run/adapter --out results/models/llama31_8b/finetuned_hybrid
for folder in Path('results/models/llama31_8b').iterdir():
    dest=ROOT/folder.name; dest.mkdir(parents=True,exist_ok=True)
    for p in folder.glob('raw_*.jsonl'): shutil.copy2(p,dest/p.name)
print('Llama outputs copied to',ROOT)